In [ ]:
import pandas as pd, numpy as np

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
# Online Retail II dataset: 1 file (online_retail_II.xlsx) with 2 sheets
# Sheet 1: Dec 2009 - Dec 2010, Sheet 2: Dec 2010 - Dec 2011 (full period: Dec 2009 - Dec 2011)
# Gives transaction data: invoice, stock code, description, quantity, price, date, customer, country
# Provides price and sales quantity, but no views, carts, or stock availability

In [ ]:
sales = pd.read_excel("online_retail_II.xlsx")

In [ ]:
sales.head(10)

In [ ]:
print(sales.shape)

In [ ]:
# Load dataset
sales = pd.read_excel("online_retail_II.xlsx")

# Standardize column names (remove spaces)
sales.columns = [c.strip().replace(" ", "") for c in sales.columns]

# Remove cancelled invoices (start with 'C')
sales = sales[~sales["Invoice"].astype(str).str.startswith("C")]

# Keep only valid sales (positive quantity and price)
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)]

# Drop rows with missing product info and remove duplicates
sales = sales.dropna(subset=["StockCode", "Description"]).drop_duplicates()

# Convert StockCode to string for consistency
sales["StockCode"] = sales["StockCode"].astype(str)

# Create daily date column for aggregation
sales["Day"] = sales["InvoiceDate"].dt.date

# Check final rows and columns
print(sales.shape)

In [ ]:
sales.head(10)

In [ ]:
# Cleaning removed 20,731 rows (~3.95%): 525,461 -> 504,730
# Added 1 column (Day): 8 -> 9 columns

In [ ]:
# Retail Rocket dataset: 4 files (events.csv, item_properties_part1.csv,item_properties_part2.csv, category_tree.csv)
# Gives demand signals (views, add-to-carts, purchases) and stock availability
# Does NOT contain price information

In [ ]:
# Load user events (views, carts, purchases)
events = pd.read_csv("Retail Rocket/events.csv")

# Convert timestamp from milliseconds to datetime
events["timestamp"] = pd.to_datetime(events["timestamp"], unit="ms")

# Remove duplicate event records
events = events.drop_duplicates()

# Show first 10 rows of events
print(events.head(10))

# Check rows and columns of events
print(events.shape)

# Combine both item property files into one table
props = pd.concat([pd.read_csv("Retail Rocket/item_properties_part1.csv"),pd.read_csv("Retail Rocket/item_properties_part2.csv")], ignore_index=True)

# Keep only stock availability records
stock = props[props["property"] == "available"].copy()

# Free memory by deleting the large props table
del props

# Convert timestamp from milliseconds to datetime
stock["timestamp"] = pd.to_datetime(stock["timestamp"], unit="ms")

# Convert availability value (0/1) to integer
stock["available"] = stock["value"].astype(int)

# Keep needed columns and sort by item and time
stock = stock[["timestamp", "itemid", "available"]].sort_values(["itemid", "timestamp"])

# Show first 10 rows of stock
print(stock.head(10))

# Check rows and columns of stock
print(stock.shape)

In [ ]:
counts = sales.groupby(["StockCode", "Day"]).size()
multi = counts[counts > 1]

result = multi.groupby(level="StockCode").head(1).reset_index(name="rows_that_day")
result["total_entries"] = result["StockCode"].map(sales["StockCode"].value_counts())

print(result)
print("Products with multi-record days:", result.shape[0])

In [ ]:
daily = (sales.groupby(["StockCode", "Day"])
              .agg(price=("Price", "mean"), qty=("Quantity", "sum"))
              .reset_index()
              .sort_values(["StockCode", "Day"]))

print(daily["StockCode"].nunique())   # number of unique products
print(daily.sample(10))               # random rows from different products

In [ ]:
print("Total unique products:", daily["StockCode"].nunique())
print("Total unique days:", daily["Day"].nunique())
print("Total unique customers:", sales["CustomerID"].nunique())

In [ ]:
# Remove old columns left from the earlier run
daily = daily.drop(columns=["dp", "dq"], errors="ignore")

# Group daily data by product so changes are calculated within each StockCode
product_groups = daily.groupby("StockCode")

# Percentage change in price vs previous record of the same product
daily["price_change_percentage"] = product_groups["price"].pct_change()

# Percentage change in quantity vs previous record of the same product
daily["quantity_change_percentage"] = product_groups["qty"].pct_change()

# Keep rows where price changed and both changes exist (not NaN)
daily = daily[
    daily["price_change_percentage"].notna()
    & (daily["price_change_percentage"] != 0)
    & daily["quantity_change_percentage"].notna()
].reset_index(drop=True)

# Price elasticity = quantity change percentage / price change percentage 
daily["elasticity"] = (daily["quantity_change_percentage"] / daily["price_change_percentage"]).clip(-20, 20)

print(daily.shape)
print(daily.head())
print("Average elasticity:", daily["elasticity"].mean())

In [ ]:
# Find the midpoint of the event timeline (splits data into older and recent halves)
mid = events["timestamp"].min() + (events["timestamp"].max() - events["timestamp"].min()) / 2

# Count events per item in the recent half and the older half
recent = events[events["timestamp"] > mid].groupby("itemid").size().rename("recent")
older  = events[events["timestamp"] <= mid].groupby("itemid").size().rename("older")

# Combine both counts into one table (0 if an item has no events in a half)
items = pd.concat([recent, older], axis=1).fillna(0)

# Demand trend = recent / older (inf -> 2.0, NaN -> 1.0)
items["trend"] = (items["recent"] / items["older"]).replace(np.inf, 2.0).fillna(1.0)

# Latest stock status of each item (assume in stock if no record)
last_stock = stock.groupby("itemid")["available"].last().rename("in_stock")
items = items.join(last_stock).fillna({"in_stock": 1})

print(items.shape)
print(items.head())

In [ ]:
# Average price elasticity from the sales data
avg_e = daily["elasticity"].mean()

# Dampening factor: the more elastic the demand, the smaller the price change
damp = min(1, 3 / abs(avg_e))

def advise(row):
    # Item is scarce if its latest stock status is 0 (out of stock)
    scarce = row["in_stock"] == 0

    if row["trend"] > 1:        # demand rising
        change = 0.10 if scarce else 0.04
    elif row["trend"] < 1:      # demand reducing
        change = -0.02 if scarce else -0.08
    else:                       # trend == 1, demand stable
        change = 0.0

    return round(change * damp, 4)

# Apply the rule to every item
items["suggested_change"] = items.apply(advise, axis=1)

print(items["suggested_change"].value_counts())
print(items.head())

In [ ]:
def explain(row):
    trend_value = row["trend"]
    change = row["suggested_change"]

    # Describe demand using the same rule as advise()
    if trend_value > 1:
        demand = f"demand is rising ({trend_value:.1f}x recent vs earlier activity)"
    elif trend_value < 1:
        demand = f"demand is falling ({trend_value:.1f}x recent vs earlier activity)"
    else:
        demand = "demand is stable"

    # Describe stock status
    stock = "item is out of stock" if row["in_stock"] == 0 else "stock is healthy"

    # Describe the price action
    if change > 0:
        action = f"Raise price by {change * 100:.1f}%"
    elif change < 0:
        action = f"Lower price by {abs(change) * 100:.1f}%"
    else:
        action = "No price change"

    return (f"{action} because {demand} and {stock}. "
            f"Change is scaled by {damp:.2f} due to price sensitivity (elasticity {avg_e:.2f}).")

# Create an explanation for every item
items["explanation"] = items.apply(explain, axis=1)

In [ ]:
for item_id in items.head(20).index:
    print(item_id, "->", items.loc[item_id, "explanation"])

In [ ]:
#Testing

In [ ]:
# Time-based split: first 80% of days = train, last 20% = test
daily = daily.sort_values("Day")
n = int(len(daily) * 0.8)
train, test = daily.iloc[:n], daily.iloc[n:]

# Mean elasticity of each product, learned from the train part only
train_e = train.groupby("StockCode")["elasticity"].mean().rename("train_e")
test = test.join(train_e, on="StockCode").dropna(subset=["train_e"])

# Predicted and actual % change in quantity
pred = test["train_e"] * test["price_change_percentage"]
actual = test["quantity_change_percentage"]

# Ignore rows where actual change is 0 (sign 0 always counts as a miss)
valid = actual != 0

# Sign accuracy = share of rows where predicted direction matches actual direction
sign_acc = (np.sign(pred[valid]) == np.sign(actual[valid])).mean()

# MAE of the model vs baseline (always predict 0)
mae = (pred[valid] - actual[valid]).abs().mean()
baseline_mae = actual[valid].abs().mean()

print("Test rows used:", valid.sum())
print("Sign accuracy:", round(sign_acc, 3))
print("MAE:", round(mae, 3), "| Baseline MAE:", round(baseline_mae, 3))

In [ ]:
#Output --> csv file

In [ ]:
items.reset_index().to_csv("advisor_data.csv", index=False)